# Lab 06 — 08 Pytest Validation

Manual test runner for Lab 06. This notebook is **not part of the recurring Gold Job**.

Databricks Workspace folders do not support normal Python `__pycache__` writes,
so the runner stages source/tests under `/tmp` before invoking pytest.

The Spark fixture is injected directly from the active Databricks notebook
session, so no `conftest.py` dependency is required.

## 1. Resolve paths

In [0]:
import sys
import shutil
from pathlib import Path

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

source_src = lab_root / "src"
source_tests = lab_root / "tests"

if not source_src.exists():
    raise FileNotFoundError(f"src directory not found: {source_src}")
if not source_tests.exists():
    raise FileNotFoundError(f"tests directory not found: {source_tests}")

print(f"Lab root     : {lab_root}")
print(f"Source src   : {source_src}")
print(f"Source tests : {source_tests}")

## 2. Stage files under `/tmp`

In [0]:
local_root = Path("/tmp/lab06_pytest")
local_src = local_root / "src"
local_tests = local_root / "tests"

if local_root.exists():
    shutil.rmtree(local_root)

local_src.mkdir(parents=True, exist_ok=True)
local_tests.mkdir(parents=True, exist_ok=True)

for file_path in source_src.glob("*.py"):
    shutil.copy2(file_path, local_src / file_path.name)

if not (local_src / "__init__.py").exists():
    (local_src / "__init__.py").write_text("", encoding="utf-8")

for file_path in source_tests.glob("test_*.py"):
    shutil.copy2(file_path, local_tests / file_path.name)

print("Staged test files:")
for p in sorted(local_tests.glob("test_*.py")):
    print(f"  - {p.name}")

## 3. Validate test layout

In [0]:
expected = {
    "test_dimensions.py",
    "test_fact_encounters.py",
    "test_aggregations.py",
    "test_quality_rules.py",
}

discovered = {p.name for p in local_tests.glob("test_*.py")}
missing = sorted(expected - discovered)

if missing:
    raise FileNotFoundError(
        "Missing expected test files: " + ", ".join(missing)
    )

print("Test layout is complete.")

## 4. Run pytest with the active Spark session

In [0]:
import pytest

notebook_spark = spark

class SparkFixturePlugin:
    @pytest.fixture(scope="session")
    def spark(self):
        return notebook_spark

original_sys_path = list(sys.path)

# Remove already-imported Lab 06 modules so pytest definitely imports the staged
# writable copy rather than a Workspace copy.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

if str(local_root) in sys.path:
    sys.path.remove(str(local_root))
sys.path.insert(0, str(local_root))

try:
    result = pytest.main(
        [
            "-q",
            "-p",
            "no:cacheprovider",
            "--assert=plain",
            str(local_tests),
        ],
        plugins=[SparkFixturePlugin()],
    )
finally:
    sys.path[:] = original_sys_path

print("")
print(f"pytest exit code: {result}")

if result != pytest.ExitCode.OK:
    raise RuntimeError(
        f"Lab 06 pytest failed with exit code: {result}"
    )

## 5. Completion

In [0]:
print("LAB 06 — PYTEST COMPLETE")
print("Status: PASS")
print("Expected tests: 12")